# Detailliertes Finanzreporting: Apple Inc. (AAPL)

---

| | |
|---|---|
| **Unternehmen** | Apple Inc. (AAPL) |
| **Zeitraum** | 01. Januar 2020 – 31. Dezember 2024 |
| **Datenquelle** | Yahoo Finance (via `yfinance`) |
| **Sprache** | Python 3 · Jupyter Notebook |

---

## 1. Einleitung

### Analysemotivation

Apple Inc. (AAPL) wurde für diese Analyse gewählt, weil das Unternehmen mehrere zentrale Anforderungen an ein geeignetes Analyseobjekt erfüllt:

- **Krisenresilienz:** AAPL hat globale Schocks wie die COVID-19-Pandemie (2020) und den damit verbundenen Börseneinbruch von rund 30 % erfolgreich überstanden und sich anschließend stark erholt – ein idealer Stresstest für Rendite- und Risikoanalysen.
- **Datenverfügbarkeit und -qualität:** Tägliche Handelsdaten über den gesamten Zeitraum 2020–2024 sind vollständig und zuverlässig verfügbar.

## 2. Datenbeschaffung & Aufbereitung

Alle benötigten Bibliotheken werden importiert und die Rohdaten über die Yahoo Finance API geladen.

### 2.1 Bibliotheken
Die Bibliotheken `yfinance`, `pandas` und `numpy` werden für die Datenbeschaffung, -analyse und -manipulation verwendet.

In [27]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.express as px

### 2.2 Datenbeschaffung & Aufbereitung
Die Kursdaten werden über die Bibliothek `yfinance` direkt von Yahoo Finance bezogen. Der Abruf umfasst den Zeitraum vom 01. Januar 2020 bis zum 31. Dezember 2024 und enthält die täglichen OHLCV-Daten (Open, High, Low, Close, Adjusted Close, Volume). Die Spaltennamen werden zur besseren Lesbarkeit in Kleinbuchstaben umbenannt.

In [28]:
START_DATE = "2020-01-01"
END_DATE = "2024-12-31"

# Download Apple Inc. (AAPL) Daten und Aufbereitung des DataFrames
aapl = (yf.download("AAPL", start=START_DATE, end=END_DATE, auto_adjust=False)
        .reset_index()  # Datum als Spalte statt Index
        .rename(columns={
            "Date": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adjusted",
            "Volume": "volume"})
        )

# MultiIndex-Spalten von yfinance auf einfache Spaltennamen reduzieren
aapl.columns = [col[0] for col in aapl.columns]

[*********************100%***********************]  1 of 1 completed


 Ausgabe der ersten Zeilen des DataFrames in tabellarischer Form.

In [29]:
aapl

,date,adjusted,close,high,low,open,volume
0,2020-01-02,72.400520,75.087502,75.150002,73.797501,74.059998,135480400
1,2020-01-03,71.696648,74.357498,75.144997,74.125000,74.287498,146322800
2,2020-01-06,72.267944,74.949997,74.989998,73.187500,73.447502,118387200
3,2020-01-07,71.928055,74.597504,75.224998,74.370003,74.959999,108872000
4,2020-01-08,73.085114,75.797501,76.110001,74.290001,74.290001,132079200
...,...,...,...,...,...,...,...
1252,2024-12-23,253.883118,255.270004,255.649994,253.449997,254.770004,40858800
1253,2024-12-24,256.797180,258.200012,258.209991,255.289993,255.490005,23234700
1254,2024-12-26,257.612732,259.019989,260.100006,257.630005,258.190002,27237100
1255,2024-12-27,254.201370,255.589996,258.700012,253.059998,257.829987,42355300


In [30]:
fig = px.line(
    aapl,
    x="date",
    y="adjusted",
    title="Apple Inc. (AAPL) – Adjustierter Schlusskurs 2020–2024",
    labels={"date": "Datum", "adjusted": "Kurs (USD)"}
)

fig.update_layout(
    xaxis_title="Datum",
    yaxis_title="Kurs (USD)",
    hovermode="x unified"
)

fig.show()

## 3. Datenqualität & -bereinigung

Vor der eigentlichen Analyse wird der Datensatz auf mögliche Qualitätsprobleme geprüft. Konkret wird der Schlusskurs (`close`) auf drei häufige Datenfehler untersucht:

- **Fehlende Werte (`NaN`):** Tage ohne Kursangabe, z. B. durch API-Fehler oder Handelsunterbrechungen.
- **Nullwerte:** Ein Kurs von 0 ist ökonomisch nicht plausibel und deutet auf einen Datenfehler hin.
- **Negative Werte:** Aktienkurse können definitionsgemäß nicht negativ sein.

In [31]:
issues_before = {
    'missing': aapl['close'].isna().sum().item(), # Item extrahiert den skalaren Wert
    'zeros': (aapl['close'] == 0).sum().item(),
    'negatives': (aapl['close'] < 0).sum().item()
}

In [32]:
print(f"Fehlende Werte: {issues_before['missing']}")
print(f"Nullwerte: {issues_before['zeros']}")
print(f"Negative Werte: {issues_before['negatives']}")

Fehlende Werte: 0
Nullwerte: 0
Negative Werte: 0


Das Ergebnis zeigt, dass der Datensatz keine dieser Anomalien enthält – die Daten sind vollständig und konsistent und können ohne weitere Bereinigung verwendet werden.